# Лабораторная работа №1

## 1. Регулярные выражения

____


In [ ]:
import re
import html
from pathlib import Path
from docx import Document

DATA_DIR = Path('data')

## 1.1. Выделить автомобильные номера

___


In [ ]:
LETTERS = 'АВЕКМНОРСТУХ'
regexp = rf'(?<!\w)[{LETTERS}][0-9]{{3}}[{LETTERS}]{{2}}(?!\w)'

test_string = 'РТУ220О KE800К П227НА С227НА Е199КБ Е512KM О503ДП Р59ОТС A771BE Х281ОН 112МНО Т51РУ Е390МВ М9000РТ М900РТУ В5РСХИ'
assert re.findall(regexp, test_string) == ['С227НА', 'Х281ОН', 'Е390МВ']
print('Исходный тест 1.1 пройден:', re.findall(regexp, test_string))


In [ ]:
PLATE_LETTERS = LETTERS + 'ABEKMHOPCTYX'
SEP = r'[ \t/-]*'
REGION = r'(?:[1-9][0-9]{1,2}|0[1-9])'

car_pattern = (
    rf'[{PLATE_LETTERS}]{SEP}(?!000)[0-9]{{3}}'
    rf'{SEP}[{PLATE_LETTERS}]{{2}}(?:{SEP}{REGION})?'
)
trailer_pattern = rf'[{PLATE_LETTERS}]{{2}}{SEP}(?!0000)[0-9]{{4}}{SEP}{REGION}'

plate_pattern = re.compile(
    rf'(?<!\w)(?:{car_pattern}|{trailer_pattern})(?!\w|{SEP}[0-9])',
    flags=re.IGNORECASE,
)

def find_plates(text):
    return plate_pattern.findall(text)


### Поиск номеров в DOCX


In [ ]:
def read_docx_paragraphs(path):
    return [paragraph.text for paragraph in Document(path).paragraphs]

files = ['pokupka_avtomobilya_regex.docx', 'Ода дорожному потоку.docx']
documents = {name: read_docx_paragraphs(DATA_DIR / name) for name in files}


In [ ]:
for name, paragraphs in documents.items():
    text = '\n'.join(paragraphs)
    numbers = find_plates(text)
    print(f'\n{name}')
    print(f'Найдено номеров: {len(numbers)}')
    for number in numbers:
        print(number)


## 1.2. Почистить текст от HTML-тегов


In [ ]:
def clean_func(text):
    text = re.sub(r'</?(?:p|div|br|h[1-6])\b[^>]*>', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'<[^>]*>', '', text)
    text = html.unescape(text)
    return re.sub(r'\s+', ' ', text).strip()


### Исходный HTML и тесты задания


In [ ]:
text = (DATA_DIR / 'moviescripts.html').read_text(encoding='utf-8')
print(text)

In [ ]:
result = clean_func(text)
print(result)

In [ ]:
assert '\n' not in result, 'Результат должен выдаваться одной строкой!'
assert result[700:800] == 'мые прекрасные, остаются только идеями и сами по себе ценятся недорого. Иное дело — идеи, разработан'
assert result[-100:] == 'ты должны научиться спокойно расставаться со своими детищами и переключаться на настоящее и будущее.'
print('Исходные тесты 1.2 пройдены.')
print('Начало очищенного текста:', result[:250])


### Пример на тексте DOCX


In [ ]:
for name, paragraphs in documents.items():
    generated_html = ''.join(
        f'<p>{html.escape(paragraph)}</p>' for paragraph in paragraphs
    )
    cleaned = clean_func(generated_html)
    print(f'\n{name}')
    print(cleaned[:350])


## 1.3. Собрать предложение одним выражением с `re.sub`


In [ ]:
def make_sentence(tokens):
    return re.sub(r' ([,.:;!?…)\]}»])|([(\[{«]) ', r'\1\2', ' '.join(tokens))

assert make_sentence(['Привет', ',', 'как', 'дела', '?']) == 'Привет, как дела?'
text = 'Впервые передо мной забрезжил свет потенциальной осмысленности регэкспов: http://docs.python.org/dev/howto/regex.html понятно и доходчиво объясняет их функцию в роли трафарета, — использование групп для выделения конкретных участков текста.'
tokens = ['Впервые', 'передо', 'мной', 'забрезжил', 'свет', 'потенциальной', 'осмысленности', 'регэкспов', ':', 'http://docs.python.org/dev/howto/regex.html', 'понятно', 'и', 'доходчиво', 'объясняет', 'их', 'функцию', 'в', 'роли', 'трафарета', ',', '—', 'использование', 'групп', 'для', 'выделения', 'конкретных', 'участков', 'текста', '.']
assert make_sentence(tokens) == text
print(tokens, text, sep="\n")

### Пример на тексте DOCX


In [ ]:
for name, paragraphs in documents.items():
    docx_text = '\n'.join(paragraphs)
    docx_tokens = re.findall(r'\w+|[^\w\s]', docx_text)
    sentence = make_sentence(docx_tokens)
    print(f'\n{name}')
    print(sentence[:350])
